In [39]:
from calendar import day_name
from webbrowser import open_new

import pandas as pd
import numpy as np
from datetime import timedelta


In [40]:
customers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_items_dataset.csv')
order_payment = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_payments_dataset.csv')
order_review = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_sellers_dataset.csv')
products_category_name_translation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/product_category_name_translation.csv')


In [41]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

In [42]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [43]:
zip_state = geolocation[['geolocation_zip_code_prefix','geolocation_state']].drop_duplicates()
zip_state.sample(10)

,geolocation_zip_code_prefix,geolocation_state
397107,19041,SP
751079,64076,PI
476399,24422,RJ
745339,61932,CE
611101,36160,MG
909317,88380,SC
793339,73370,DF
760594,65665,MA
331822,14169,SP
856299,82590,PR


In [44]:
zip_state['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(8)

In [45]:
zip_state[
    zip_state['geolocation_zip_code_prefix'].duplicated(keep=False)
].sort_values('geolocation_zip_code_prefix')

,geolocation_zip_code_prefix,geolocation_state
21728,2116,SP
22261,2116,RN
71032,4011,SP
72852,4011,AC
430342,21550,RJ
430999,21550,AC
459234,23056,RJ
460406,23056,AC
792362,72915,GO
792394,72915,DF


In [50]:
customer_state_count = customers['customer_state'].value_counts().sort_values(ascending=False)
customer_state_count

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [51]:
total_customer_count = customers['customer_state'].value_counts().sum()
total_customer_count

np.int64(99441)

In [52]:
customer_state_per = (customer_state_count/total_customer_count)*100
customer_state_per

customer_state
SP    41.980672
RJ    12.924247
MG    11.700405
RS     5.496727
PR     5.073360
SC     3.657445
BA     3.399000
DF     2.152030
ES     2.044428
GO     2.031355
PE     1.661287
CE     1.343510
PA     0.980481
MT     0.912099
MA     0.751199
MS     0.719019
PB     0.539013
PI     0.497783
RN     0.487726
AL     0.415322
SE     0.351967
TO     0.281574
RO     0.254422
AM     0.148832
AC     0.081455
AP     0.068382
RR     0.046259
Name: count, dtype: float64

In [70]:
orders_customers = customers.merge(orders, on='customer_id', how='inner')
orders_customers_order_items = orders_customers.merge(order_items, on='order_id', how='inner')
orders_customers_order_items.groupby('customer_state')['price'].sum().sort_values(ascending=False)

customer_state
SP    5202955.05
RJ    1824092.67
MG    1585308.03
RS     750304.02
PR     683083.76
SC     520553.34
BA     511349.99
DF     302603.94
GO     294591.95
ES     275037.31
PE     262788.03
CE     227254.71
PA     178947.81
MT     156453.53
MA     119648.22
MS     116812.64
PB     115268.08
PI      86914.08
RN      83034.98
AL      80314.81
SE      58920.85
TO      49621.74
RO      46140.64
AM      22356.84
AC      15982.95
AP      13474.30
RR       7829.43
Name: price, dtype: float64

In [88]:
state_aov = orders_customers_order_items.groupby('customer_state').agg(total_revenue=('price', 'sum'),unique_order=('order_id', 'nunique')).reset_index()
state_aov['Aov'] =  state_aov['total_revenue'] / state_aov['unique_order']
state_aov = state_aov.sort_values(by='Aov', ascending=False)
state_aov

,customer_state,total_revenue,unique_order,Aov
14,PB,115268.08,532,216.669323
3,AP,13474.30,68,198.151471
0,AC,15982.95,81,197.320370
1,AL,80314.81,411,195.413163
20,RO,46140.64,247,186.804211
13,PA,178947.81,970,184.482278
26,TO,49621.74,279,177.855699
16,PI,86914.08,493,176.296308
12,MT,156453.53,903,173.259723
19,RN,83034.98,482,172.271743


In [91]:
# Task 1: Customer share by state
customer_share = customers['customer_state'].value_counts(normalize=True) * 100
# Task 2: Revenue by state
state_revenue = orders_customers_order_items.groupby('customer_state')['price'].sum()
# Task 3: Revenue share by state
revenue_share = (state_revenue / state_revenue.sum()) * 100
# Task 4: Combine customer share and revenue share
state_comparison = pd.DataFrame({'customer_share': customer_share,'revenue_share': revenue_share})
# Task 5: Calculate difference
state_comparison['difference'] = (state_comparison['revenue_share']- state_comparison['customer_share'])
# Task 6: Sort by highest difference
state_comparison = state_comparison.sort_values(by='difference',ascending=False)
# Display result
state_comparison

,customer_share,revenue_share,difference
customer_state,,,
RJ,12.924247,13.420692,0.496446
BA,3.399000,3.762238,0.363238
PA,0.980481,1.316602,0.336121
CE,1.343510,1.672018,0.328508
PB,0.539013,0.848081,0.309067
PE,1.661287,1.933453,0.272166
MT,0.912099,1.151101,0.239002
AL,0.415322,0.590913,0.175592
SC,3.657445,3.829951,0.172506


In [94]:
top_positive = state_comparison.sort_values(by='difference',ascending=False).head(5)
top_negative = state_comparison.sort_values(by='difference',ascending=True).head(5)
top_positive

,customer_share,revenue_share,difference
customer_state,,,
RJ,12.924247,13.420692,0.496446
BA,3.399000,3.762238,0.363238
PA,0.980481,1.316602,0.336121
CE,1.343510,1.672018,0.328508
PB,0.539013,0.848081,0.309067


In [95]:
top_negative

,customer_share,revenue_share,difference
customer_state,,,
SP,41.980672,38.280543,-3.700129
PR,5.073360,5.025763,-0.047597
MG,11.700405,11.663843,-0.036562
ES,2.044428,2.023577,-0.020852
RR,0.046259,0.057605,0.011346


In [102]:
state_revenue = orders_customers_order_items.groupby('customer_state')['price'].sum()
revenue_per_customer = (state_revenue / customer_state_count).sort_values(ascending=False).reset_index()
# revenue_per_customer
revenue_per_customer.columns = ['State', 'Revenue per Customer']
revenue_per_customer

,State,Revenue per Customer
0,PB,215.052388
1,AP,198.151471
2,AC,197.320370
3,AL,194.466852
4,PA,183.536215
5,RO,182.374071
6,TO,177.220500
7,PI,175.584000
8,MT,172.495623
9,RN,171.206144
